 # **Cleaning Data**

## **Objective**
Demonstrate professional-level data cleaning skills by taking a deliberately messy dataset and systematically transforming it into a clean, analysis-ready dataset. Document every decision.

### **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

### **Load Dataset**

In [ ]:
df = pd.read_csv("dirty_cafe_sales.csv")

### **Explore Data**

In [ ]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [ ]:
df.shape

(10000, 8)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [ ]:
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


### **Data Quality Report**

This section examines the dataset for missing values, duplicate records, invalid values, data type issues, and potential range anomalies before cleaning.

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Missing Values per Column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Duplicate Rows:
0

Data Types:
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object


In [ ]:
before_rows = 10000
before_nulls = 6826
before_duplicates = 0

In [ ]:
invalid_values = ["ERROR", "UNKNOWN"]

for column in df.columns:
    invalid_count = df[column].isin(invalid_values).sum()
    print(f"{column}: {invalid_count} invalid values")

Transaction ID: 0 invalid values
Item: 636 invalid values
Quantity: 341 invalid values
Price Per Unit: 354 invalid values
Total Spent: 329 invalid values
Payment Method: 599 invalid values
Location: 696 invalid values
Transaction Date: 301 invalid values


### **Handling Invalid Values**

Invalid entries in the form of “ERROR” and “UNKNOWN” appear in our dataset. Such invalid entries cannot be used for further processing and can affect the process of converting into other data types. Therefore, they will be treated as missing values (NaN).

In [ ]:
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

In [ ]:
print(df.isnull().sum())

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


### **Correction of Data Types**

Many columns are saved in object form due to the presence of bad values and inconsistencies within the dataset. Upon conversion of invalid values into missing values, proper data types can then be assigned to make sure that the dataset is ready for analysis.

In [ ]:
df["Transaction ID"] = df["Transaction ID"].astype("string")

numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

df.dtypes

,0
Transaction ID,string[python]
Item,object
Quantity,float64
Price Per Unit,float64
Total Spent,float64
Payment Method,object
Location,object
Transaction Date,datetime64[ns]


### **Dealing With Missing Data**

Missing data techniques are used depending upon the type of the columns. For numeric columns, median imputation technique is used as median is not very sensitive to outliers. The mode imputation method is used for categorical columns since it denotes the frequently occurring value. Rows where the transaction date is missing are deleted as it is not possible to find out the exact date.

In [ ]:
df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [ ]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

In [ ]:
categorical_columns = ["Item", "Payment Method", "Location"]

for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

In [ ]:
df = df.dropna(subset=["Transaction ID"])

In [ ]:
df = df.dropna(subset=["Transaction Date"])

In [ ]:
print(df.isnull().sum())

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


### **Checking for Duplicates and Deleting Them**

To ensure that duplicate transactions do not influence our analysis, duplicate rows were identified and deleted.

In [ ]:
duplicates_before = df.duplicated().sum()

print("Duplicate rows before removal:", duplicates_before)

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print("Duplicate rows after removal:", duplicates_after)

print("Duplicates removed:", duplicates_before - duplicates_after)

Duplicate rows before removal: 0
Duplicate rows after removal: 0
Duplicates removed: 0


### **Standardization of Data**

Text variables were standardized by deleting additional spaces and using a standardized format. This will help to store values for identical categories in a uniform way.

In [ ]:
df["Item"] = df["Item"].str.strip().str.title()

df["Payment Method"] = df["Payment Method"].str.strip().str.title()

df["Location"] = df["Location"].str.strip().str.title()

In [ ]:
print("Item:")
print(df["Item"].unique())

print("\nPayment Method:")
print(df["Payment Method"].unique())

print("\nLocation:")
print(df["Location"].unique())

Item:
['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'Juice' 'Sandwich' 'Tea']

Payment Method:
['Credit Card' 'Cash' 'Digital Wallet']

Location:
['Takeaway' 'In-Store']


### **Detecting Outliers**

The Interquartile Range (IQR) technique was applied to detect any outliers in the numeric columns. The identified outliers were analyzed to see if they were true transactions or just input mistakes.

In [ ]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ].shape[0]

    print(f"{column}")
    print(f"Lower Bound: {lower_bound}")
    print(f"Upper Bound: {upper_bound}")
    print(f"Number of Outliers: {outlier_count}")
    print("-" * 40)

Quantity
Lower Bound: -1.0
Upper Bound: 7.0
Number of Outliers: 0
----------------------------------------
Price Per Unit
Lower Bound: -1.0
Upper Bound: 7.0
Number of Outliers: 0
----------------------------------------
Total Spent
Lower Bound: -8.0
Upper Bound: 24.0
Number of Outliers: 250
----------------------------------------


### **Value Range Validation**

Validation was done to ensure that there were no invalid values within the numeric columns, such as negative figures.

In [ ]:
for column in numeric_columns:
    print(f"\n{column}")
    print("Minimum:", df[column].min())
    print("Maximum:", df[column].max())


Quantity
Minimum: 1.0
Maximum: 5.0

Price Per Unit
Minimum: 1.0
Maximum: 5.0

Total Spent
Minimum: 1.0
Maximum: 25.0


In [ ]:
for column in numeric_columns:
    negative_values = (df[column] < 0).sum()
    print(f"{column}: {negative_values} negative values")

Quantity: 0 negative values
Price Per Unit: 0 negative values
Total Spent: 0 negative values


In [ ]:
Q1 = df["Total Spent"].quantile(0.25)
Q3 = df["Total Spent"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

total_spent_outliers = df[
    (df["Total Spent"] < lower_bound) |
    (df["Total Spent"] > upper_bound)
]

total_spent_outliers.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
10,TXN_2548360,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-11-07
51,TXN_6342161,Salad,5.0,5.0,25.0,Digital Wallet,Takeaway,2023-01-08
52,TXN_8914892,Juice,5.0,5.0,25.0,Digital Wallet,Takeaway,2023-03-15
96,TXN_5220895,Salad,5.0,5.0,25.0,Cash,In-Store,2023-06-10
100,TXN_9517146,Juice,5.0,5.0,25.0,Cash,Takeaway,2023-10-30
150,TXN_8687151,Salad,5.0,5.0,25.0,Cash,Takeaway,2023-06-10
157,TXN_4283157,Salad,5.0,5.0,25.0,Digital Wallet,In-Store,2023-11-25
177,TXN_1896955,Salad,3.0,5.0,25.0,Digital Wallet,In-Store,2023-09-23
214,TXN_8693704,Salad,3.0,5.0,25.0,Cash,In-Store,2023-05-04
330,TXN_5523450,Salad,5.0,5.0,25.0,Credit Card,Takeaway,2023-07-02


### **Numeric Range Validation**

The numeric columns were checked for unrealistic values, including negative quantities, negative prices, and negative transaction amounts.

In [ ]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]

for column in numeric_columns:
    print(f"\n{column}")
    print("Minimum:", df[column].min())
    print("Maximum:", df[column].max())

    negative_values = (df[column] < 0).sum()
    print("Negative values:", negative_values)


Quantity
Minimum: 1.0
Maximum: 5.0
Negative values: 0

Price Per Unit
Minimum: 1.0
Maximum: 5.0
Negative values: 0

Total Spent
Minimum: 1.0
Maximum: 25.0
Negative values: 0


### **Final Data Quality Check**

Following the completion of the data cleaning step, the dataset was evaluated for missing data, duplicate data, and proper data types.

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

print("\nDataset Shape:")
print(df.shape)

Missing Values:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Duplicate Rows:
0

Data Types:
Transaction ID      string[python]
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

Dataset Shape:
(9540, 8)


### **Before vs After Summary Table**

The table below compares the overall data quality of the dataset before and after the cleaning process.

In [ ]:
after_rows = df.shape[0]
after_nulls = df.isnull().sum().sum()
after_duplicates = df.duplicated().sum()

summary = pd.DataFrame({
    "Metric": [
        "Row Count",
        "Total Missing Values",
        "Duplicate Rows",
        "Data Type Accuracy"
    ],
    "Before Cleaning": [
        10000,
        6826,
        0,
        "Incorrect"
    ],
    "After Cleaning": [
        after_rows,
        after_nulls,
        after_duplicates,
        "Correct"
    ]
})

summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,10000,9540
1,Total Missing Values,6826,0
2,Duplicate Rows,0,0
3,Data Type Accuracy,Incorrect,Correct


### **Store Clean Data**

The dataset after cleaning and preparation for analysis was stored in a new CSV file.

In [ ]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


## **Conclusion**

For this project the messy café sales dataset was cleaned and transformed into a dataset suitable for analysis using Python, pandas and NumPy.

The assessment of data quality found that there were missing values, invalid placeholder values, wrong data types, and possible outliers. Before carrying out the suitable cleaning procedures, invalid values such as "ERROR" and "UNKNOWN" were changed into missing values. Numeric columns were dealt with by using median imputation, categorical columns by using mode imputation, and the rows which had missing transaction dates were deleted since the correct dates could not be reliably deduced.

Duplicates have been checked and no exact duplicate rows have been found. The text-based columns have been standardised to ensure consistent formatting, whereas the numeric columns and the date columns have been converted into their appropriate data types. The IQR method was applied in order to identify possible outliers, and the Total Spent values thus identified have been kept since they were plausible transaction amounts.

As a result of the cleaning process, the dataset was reduced in size from 10,000 rows to 9,540 rows and had no missing values or duplicate entries left. The final dataset is now clean, consistent and ready for further analysis and visualisation.

The project shows how important it is to carry out systematic data cleaning before analysis, since better data quality leads to more reliable and meaningful insights.